In [7]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [8]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
89,"Although I'm a girl, thankfully I have a sense...",positive
588,ok we have a film that some are calling one of...,negative
598,"A mix of comedy, romance, music(?!), action an...",positive
397,Trapped: buried alive brings us to a resort th...,positive
858,Being an unrelenting non-stop over-the-top exp...,negative


In [9]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [10]:
df = normalize_text(df)
df.head()

,review,sentiment
89,although girl thankfully sense humor realize r...,positive
588,ok film calling one best movie ever but sittin...,negative
598,mix comedy romance music action horror knockou...,positive
397,trapped buried alive brings u resort opened so...,positive
858,unrelenting non stop over the top explosive me...,negative


In [11]:
df['sentiment'].value_counts()

sentiment
negative    266
positive    234
Name: count, dtype: int64

In [12]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [13]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
89,although girl thankfully sense humor realize r...,1
588,ok film calling one best movie ever but sittin...,0
598,mix comedy romance music action horror knockou...,1
397,trapped buried alive brings u resort opened so...,1
858,unrelenting non stop over the top explosive me...,0


In [14]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [15]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [17]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/fayyazsavanur1/MLOPS-Capstone-Project.mlflow')
dagshub.init(repo_owner='fayyazsavanur1', repo_name='MLOPS-Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("NLP IMDB Rating")


Initialized MLflow to track repo "fayyazsavanur1/MLOPS-Capstone-Project"

2026-08-16 13:52:40,962 - INFO - Initialized MLflow to track repo "fayyazsavanur1/MLOPS-Capstone-Project"


Repository fayyazsavanur1/MLOPS-Capstone-Project initialized!

2026-08-16 13:52:40,965 - INFO - Repository fayyazsavanur1/MLOPS-Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/4f4b16bcc82a42229e96563840033790', creation_time=1786852095615, experiment_id='0', last_update_time=1786852095615, lifecycle_stage='active', name='NLP IMDB Rating', tags={}>

In [19]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-08-16 13:53:55,929 - INFO - Starting MLflow run...
2026-08-16 13:53:56,807 - INFO - Logging preprocessing parameters...
2026-08-16 13:53:57,829 - INFO - Initializing Logistic Regression model...
2026-08-16 13:53:57,830 - INFO - Fitting the model...
2026-08-16 13:53:57,867 - INFO - Model training complete.
2026-08-16 13:53:57,870 - INFO - Logging model parameters...
2026-08-16 13:53:58,216 - INFO - Making predictions...
2026-08-16 13:53:58,220 - INFO - Calculating evaluation metrics...
2026-08-16 13:53:58,238 - INFO - Logging evaluation metrics...
2026-08-16 13:53:59,372 - INFO - Saving and logging the model...
f:\Youtube\MLOPS\11. Capstone Project\MLOPS-Capstone-Project\.venv\lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptoo